# Finsler-Adam: Quick Demo on CIFAR-10

This notebook demonstrates Finsler-Adam vs AdamW on CIFAR-10 / ResNet-20.
Run all cells — results appear in ~5 minutes on a free Colab GPU.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsukuyu/finsler-adam/blob/main/notebooks/finsler_adam_demo.ipynb)

In [ ]:
# Install Finsler-Adam from GitHub (not yet on PyPI)
!pip install -q "git+https://github.com/Tsukuyu/finsler-adam.git" torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
from finsler_adam import FinslerAdam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Data & Model

In [ ]:
# CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_test)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
# ResNet-20 for CIFAR
class BasicBlock(nn.Module):
    def __init__(self, in_p, out_p, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_p, out_p, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_p)
        self.conv2 = nn.Conv2d(out_p, out_p, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_p)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_p != out_p:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_p, out_p, 1, stride, bias=False), nn.BatchNorm2d(out_p))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.shortcut(x))

class ResNet20(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._layer(16, 16, 3, 1)
        self.layer2 = self._layer(16, 32, 3, 2)
        self.layer3 = self._layer(32, 64, 3, 2)
        self.fc = nn.Linear(64, 10)
    def _layer(self, inp, out, n, s):
        return nn.Sequential(*[BasicBlock(inp if i==0 else out, out, s if i==0 else 1) for i in range(n)])
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

print(f'ResNet-20 params: {sum(p.numel() for p in ResNet20().parameters())/1e6:.2f}M')

## 2. Train & Compare

In [ ]:
EPOCHS = 50
CONFIGS = {
    'AdamW':        dict(gamma=0.0, anna_alpha=0.0),
    'Finsler-Adam': dict(gamma=0.5, anna_alpha=0.1),
    'AnnaOnly':     dict(gamma=0.0, anna_alpha=0.1),
}

results = {}

for name, cfg in CONFIGS.items():
    print(f'\n=== {name} ===')
    torch.manual_seed(42)
    model = ResNet20().to(device)
    opt = FinslerAdam(model.parameters(), lr=1e-3, weight_decay=0.01, **cfg)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    
    train_losses, val_accs = [], []
    for ep in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        sched.step()
        train_losses.append(total_loss / len(trainloader))
        
        model.eval()
        correct = sum((model(x.to(device)).argmax(1) == y.to(device)).sum().item()
                      for x, y in testloader)
        val_accs.append(100 * correct / len(testset))
        
        if ep % 10 == 0:
            print(f'  Ep {ep:3d} | Loss {train_losses[-1]:.4f} | Val Acc {val_accs[-1]:.1f}%')
    
    results[name] = dict(train_loss=train_losses, val_acc=val_accs)
    print(f'  Final: {val_accs[-1]:.2f}%')

## 3. Plot Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'AdamW': '#1f77b4', 'Finsler-Adam': '#d62728', 'AnnaOnly': '#2ca02c'}

for name, data in results.items():
    ax1.plot(data['train_loss'], label=name, color=colors[name], linewidth=2)
    ax2.plot(data['val_acc'], label=name, color=colors[name], linewidth=2)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss')
ax1.set_title('CIFAR-10 / ResNet-20: Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation Accuracy (%)')
ax2.set_title('CIFAR-10 / ResNet-20: Validation Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('finsler_adam_cifar10.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to finsler_adam_cifar10.png')

## 4. Anna-Limit Response Curve

In [ ]:
import numpy as np
from finsler_adam import anna_clip

g = torch.linspace(-10, 10, 1000)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(g.numpy(), g.numpy(), '--', color='gray', alpha=0.5, label='Identity (no clip)')
for alpha in [0.05, 0.1, 0.3]:
    clipped = anna_clip(g, alpha=alpha)
    ax.plot(g.numpy(), clipped.numpy(), linewidth=2, label=f'Anna-Limit (α={alpha})')
ax.set_xlabel('Input gradient'); ax.set_ylabel('Output gradient')
ax.set_title('Anna-Limit: Smooth Gradient Clipping (4/3 exponent)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()